# 04 — BOM irradiance: probe, map, extract

**Deliverable D12a.** The only step in this project that touches AWS, and it runs once.

Everything here reuses what already exists rather than reinventing it:

| | |
|---|---|
| Athena helper | `bms_sa_review.shared.aws_config.aq` — same SSO profile `ciccada`, same S3 staging |
| Table + access pattern | `bom_nci.solar`, lifted from `build_structured_data.py` |
| Postcode geometry | the `sjoin` approach from `BOM_NCI/Get_ALL_postcodes_ABS.ipynb` |

**The one genuinely missing piece.** Solar Analytics stored `n_lat` / `n_long` per site
in `meta_up23c`. SolarEdge gives a postcode and nothing else, so the grid points have to
be derived: postcode → ABS POA-2021 polygon → the BOM nodes inside it.

**Prerequisite:** `aws sso login --profile ciccada`

**Cost discipline.** Athena bills by data scanned and `bom_nci.solar` is large. The
probes below scan almost nothing and answer the two questions that decide whether the
extract is worth running at all. Run them first — that is the entire point of this
notebook's ordering.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_metadata as meta, se_bom, se_params
from solar_edge.lib import se_counterfactual as ctf   # section 5 builds the counterfactual

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG      # section 5 scores against the same cohort as 02/03
print("POA shapefile:", C.POA_SHAPEFILE, "->", C.POA_SHAPEFILE.exists())

POA shapefile: C:\Users\z3553082\OneDrive - UNSW\Documents\CICCADA - Data\POA_2021_AUST_GDA2020_SHP\POA_2021_AUST_GDA2020.shp -> True


## 1. Probe — in cost order, cheapest first

**Read this before running anything.** `bom_nci.solar` covers the whole Himawari disc
at 10-minute resolution. An unbounded query against it is not cheap, and an earlier
version of this notebook had one — `probe_coverage` scanned a full year across the
entire grid with two `count(DISTINCT ...)` aggregates and no spatial filter. That is
the query to avoid.

Everything below is now bounded to `FLEET_BOX` (NSW/SA/QLD plus margin) and to a short
time range, and they run in this order deliberately:

| step | scope | purpose |
|---|---|---|
| `describe_bom()` | metadata only | is the table partitioned, and on what? |
| `probe_one_day()` | 1 day, fleet box | does 2025 exist? what does a row look like? |
| `probe_coverage()` | 1 month by default | density and grid stability |
| `probe_grid()` | 1 month, fleet box | how many nodes → how big is the extract |

**`describe_bom()` is the important one.** If the table has `year`/`month` partition
columns, filters must use *those*. `WHERE year(time) = 2025` applies a function to the
column, which Athena cannot use for partition pruning, so it scans everything regardless
of the filter. Worth noting `build_structured_data.py` does exactly that, so the existing
pipeline may be scanning more than it needs to as well.

If `probe_one_day` is slow, stop and look at the partitioning before going further.

In [2]:
aq = se_bom.get_aq()          # raises with instructions if SSO isn't active

# 1a. Metadata only — instant, free. Look for partition columns.
display(se_bom.describe_bom(aq))

""


In [3]:
# 1b. One day, fleet box. If this is slow, the table is likely unpartitioned
# (or not pruning) — investigate before running anything wider.
import time
t0 = time.perf_counter()
display(se_bom.probe_one_day(aq, day="2025-06-15"))
print(f"elapsed: {time.perf_counter() - t0:.1f}s")

,n_rows,n_grid_points,n_times,first_time,last_time,mean_ghi
0,8294517,139639,74,2025-06-15,2025-06-15 23:50:00,284.494359


elapsed: 4.3s


In [4]:
# 1c. One month. Widen with months=[1,2,...] only once you know what 1c costs.
t0 = time.perf_counter()
coverage = se_bom.probe_coverage(aq, months=[6])
display(coverage)
print(f"elapsed: {time.perf_counter() - t0:.1f}s")
print(f"n_times: {int(coverage.n_times.iloc[0]):,}  (expect ~4,320 for a 30-day month)")
print(f"grid points in fleet box: {int(coverage.n_grid_points.iloc[0]):,}")

,year,month,n_rows,n_times,n_grid_points,first_time,last_time
0,2025,6,249189661,2226,139639,2025-06-01,2025-06-30 23:50:00


elapsed: 13.0s
n_times: 2,226  (expect ~4,320 for a 30-day month)
grid points in fleet box: 139,639


**`probe_grid`** sizes the extract. Rows ≈ `n_grid_points` × 52,560 ten-minute slots per
year. At ~400 nodes that's ~21 M rows and a few hundred MB; at 2,000 nodes it's ~105 M
rows and needs chunking. That is the difference between a comfortable local file and a
different plan.

It also settles `se_config.BOM_GRID_SPACING_DEG`, which is currently an **unverified
guess** inferred from `process_bom.ipynb` rounding coordinates to 2 decimal places.

In [5]:
grid_box = se_bom.probe_grid(aq)
display(grid_box)

n = int(grid_box.n_grid_points.iloc[0])
print(f"Estimated extract: {n:,} nodes x 52,560 slots = {n * 52560 / 1e6:,.0f} M rows")
print(f"Rough Parquet size: {n * 52560 * 16 / 1024**3:,.1f} GB uncompressed, "
      f"perhaps {n * 52560 * 16 / 1024**3 / 5:,.1f} GB compressed")

,n_grid_points,n_distinct_lat,n_distinct_lon,lat_min,lat_max,lon_min,lon_max
0,169822,1538,1808,-43.56,-10.04,113.4,153.64


Estimated extract: 169,822 nodes x 52,560 slots = 8,926 M rows
Rough Parquet size: 133.0 GB uncompressed, perhaps 26.6 GB compressed


In [6]:
# Read the node spacing directly instead of inferring it.
spacing = se_bom.probe_spacing(aq)
display(spacing.head(10))
if len(spacing) > 1:
    step = spacing.latitude.diff().dropna().round(4).mode().iloc[0]
    print(f"Observed latitude step: {step} degrees")
    print(f"se_config.BOM_GRID_SPACING_DEG is currently {C.BOM_GRID_SPACING_DEG} - update if these differ.")

,latitude
0,-38.00
1,-37.98
2,-37.96
3,-37.94
4,-37.92
5,-37.90
6,-37.88
7,-37.86
8,-37.84
9,-37.82


Observed latitude step: 0.02 degrees
se_config.BOM_GRID_SPACING_DEG is currently 0.02 - update if these differ.


## 2. Attach geography to the site dimension

Postcode → ABS POA-2021 polygon → representative point and polygon area.

`representative_point()` rather than the geometric centroid, because a centroid can fall
outside a concave or multi-part polygon, and Australian postcodes include plenty of both.

**`postcode_area_km2` is the column to watch.** It is the single largest methodological
compromise in this port: Solar Analytics had per-site coordinates, this has a postcode.
A regional postcode spanning tens of kilometres puts the irradiance reference well away
from the actual array, and cloud fields decorrelate over that distance. Recording the
area makes the resulting bias auditable rather than invisible.

In [7]:
enriched = meta.attach_geography(con)
print(f"Sites with geography: {enriched.centroid_lat.notna().sum():,} of {len(enriched):,}")
display(enriched[["site_alias", "state", "postcode",
                  "centroid_lat", "centroid_lon", "postcode_area_km2"]].head())

area = enriched.postcode_area_km2
print("\nPostcode polygon area (km^2):")
display(area.describe(percentiles=[.5, .75, .9, .99]).round(1))
print(f"\nSites in postcodes > 1000 km^2: {(area > 1000).sum():,} "
      f"({100 * (area > 1000).mean():.1f}%) - expect the D12c quality gate to reject "
      "these preferentially, which is correct but NOT random attrition.")

Sites with geography: 1,602 of 1,602


,site_alias,state,postcode,centroid_lat,centroid_lon,postcode_area_km2
0,AUS001,New South Wales,2291,-32.947608,151.741900,7.193861
1,AUS002,South Australia,5109,-34.775142,138.671013,14.603661
2,AUS003,South Australia,5172,-35.286253,138.615336,170.856767
3,AUS004,South Australia,5172,-35.286253,138.615336,170.856767
4,AUS005,South Australia,5051,-35.034157,138.612795,17.684241



Postcode polygon area (km^2):


count     1602.0
mean       457.0
std       1737.8
min          0.8
50%         24.3
75%        170.9
90%       1029.4
99%       5337.6
max      47733.5
Name: postcode_area_km2, dtype: float64


Sites in postcodes > 1000 km^2: 167 (10.4%) - expect the D12c quality gate to reject these preferentially, which is correct but NOT random attrition.


## 3. Map postcodes to BOM grid nodes

Spatial join, `predicate="within"`, following `Get_ALL_postcodes_ABS.ipynb`.

Postcodes containing **no** node fall back to the nearest node to their representative
point. Small urban postcodes are routinely smaller than the grid spacing, so without that
fallback a large share of the fleet — the dense metropolitan part — would silently lose
its irradiance.

Note that `process_bom.ipynb` ultimately **averages** all nodes within a postcode
(`groupby(['time','postcode']).mean()`). For SolarEdge that average is the better
estimator than snapping to one node: there is no "nearest" to snap to when the site
location inside the postcode is unknown.

In [8]:
grid_points = se_bom.fetch_grid_points(aq, con)
print(f"Distinct grid nodes in the fleet bounding box: {len(grid_points):,}")

mapping = se_bom.postcode_grid_points(con, grid_points)
display(mapping.head())

print(f"\nPostcodes mapped: {mapping.postcode.nunique():,}")
display(mapping.match_type.value_counts())
print("\nNodes per postcode:")
display(mapping.groupby("postcode").size().describe(percentiles=[.5, .9]).round(1))

mapping.to_parquet(C.STORE_DIR / "se_postcode_grid.parquet", index=False)

Distinct grid nodes in the fleet bounding box: 133,149
24 postcode(s) contain no grid node and were matched to the nearest one (median 1.0 km away).


,postcode,latitude,longitude,match_type
0,5280,-37.84,140.38,within
1,2756,-33.44,150.90,within
2,2400,-29.48,149.82,within
3,2400,-29.26,149.78,within
4,4310,-28.02,152.62,within



Postcodes mapped: 507


match_type
within     27367
nearest       24
Name: count, dtype: int64


Nodes per postcode:


count     507.0
mean       54.0
std       130.9
min         1.0
50%         6.0
90%       163.2
max      1689.0
dtype: float64

## 4. Extract

The expensive call. Read the probe output above before setting `RUN_EXTRACT = True`.

**What it does:** for each month, reads `bom_nci.solar` for the grid nodes that fall
inside a fleet postcode, averages them **to postcode inside Athena**, and appends. The
per-node values are never needed — `process_bom.ipynb` averages them too — so
aggregating at the source rather than locally cuts the transferred volume by the number
of nodes per postcode.

**Expected size:** ~507 postcodes × 4,464 ten-minute slots ≈ **2 M rows per month**,
~27 M for the year, a few hundred MB.

> **An earlier version of this was badly wrong.** It filtered on the *bounding box* of
> the grid points, which for sites across NSW, SA and QLD is most of eastern Australia —
> returning **341 million rows for January alone**, roughly 4 billion for the year. If
> you ran that, the output is not usable and the scan was expensive. It now joins on the
> explicit node list instead.

Columns match what `build_structured_data.py` consumes: `time`, `postcode`, `GHI`,
`cloud_type` — the last is what the clear-sky day selection ranks on.

> **Athena query-size limit.** `StartQueryExecution` rejects a `queryString` over
> **262,144 characters**. The postcode↔node mapping is inlined as a `VALUES` clause, and
> the full list (~9,500 nodes) is about 285 KB — over the cap. `extract_bom` now thins it
> automatically, keeping the nodes nearest each postcode's centre and reporting the
> reduction.
>
> This is not purely a workaround. Postcode 4702 alone spans ~200 km and contains hundreds
> of nodes; averaging irradiance across all of them describes a region rather than the sky
> above an inverter. Dense urban postcodes have only a few nodes and are untouched.

In [9]:
RUN_EXTRACT = True   # set True once the probes and the row estimate look right

if RUN_EXTRACT:
    # `mapping`, NOT `grid_points`. Filtering on a bare node list's bounding box
    # pulls every node in the rectangle — most of eastern Australia.
    #
    # The mapping is auto-thinned if it does not fit: Athena rejects any query
    # longer than 262,144 characters, and the full node list renders to ~285 KB
    # of inline VALUES. `fit_mapping_to_athena` keeps the nodes closest to each
    # postcode's centre and prints what it dropped. Pass
    # max_nodes_per_postcode=N to set the cap yourself.
    bom = se_bom.extract_bom(aq, mapping=mapping)
    display(bom.head())
else:
    print("RUN_EXTRACT is False — check the estimate printed above first.")

Mapping thinned to <= 32 node(s) per postcode: 27,391 -> 6,587 nodes (685 KB -> 165 KB of SQL).
  Athena caps a query at 262,144 characters; the full list does not fit.
  Nodes kept are those closest to each postcode's centre, so large rural postcodes lose their far-flung nodes and dense urban ones are unchanged.
Extracting 507 postcodes from 6,587 grid nodes, 12 month(s).
Expect roughly 2.3 M rows per month.

  2025-01: 1,297,362 rows
  2025-02: 1,108,779 rows
  2025-03: 1,140,554 rows
  2025-04: 1,014,500 rows
  2025-05: 975,055 rows
  2025-06: 907,653 rows
  2025-07: 957,558 rows
  2025-08: 1,023,449 rows
  2025-09: 1,073,477 rows
  2025-10: 1,175,176 rows
  2025-11: 1,240,677 rows
  2025-12: 1,319,695 rows

Wrote 13,233,935 rows to C:\Users\z3553082\AppData\Local\ciccada\solar_edge_store\bom_solar_2025.parquet (124 MB)


,time,postcode,GHI,cloud_type,n_nodes
0,2025-01-05 23:40:00,2642,273.492500,5.7500,32
1,2025-01-08 04:00:00,2642,988.489687,0.0000,32
2,2025-01-09 23:20:00,2642,553.986250,6.8750,32
3,2025-01-10 03:00:00,2642,839.484062,6.6875,32
4,2025-01-13 00:30:00,2642,956.246562,0.3750,32


## 5. Build the counterfactual (D12b / D12c)

**The extract above is not the counterfactual.** It lands irradiance per postcode and
stops. `se_uncurtailedpv` — the "what would this inverter have produced" table that
Method B and Volt-Watt 5b both need — is three further steps, and until they run,
notebook 06 will keep reporting *Method B not runnable yet*.

| step | in | out |
|---|---|---|
| `build_structured` | `se_interval` + `bom_solar` | `se_structured` — GHI, empirical clear-sky `GHI_cs`, normalised P |
| `fit_ghi_model` | `se_structured` | `se_ghi_model` — per site, per 5-min time-of-day bin |
| `build_uncurtailedpv` | both | `se_uncurtailedpv` — the counterfactual |

`GHI_cs` is **not** a modelled clear-sky curve. It is derived from the BOM data itself —
clearest day per postcode-month, then a percentile over a window — because the model's
regressor is the ratio `GHI / GHI_cs`. Substituting pvlib would change what that ratio
means and the coefficients would stop being comparable to the Solar Analytics ones.

### 5.0 Alignment check — run this first

`build_structured` assumes `bom_solar.time` is **UTC** and applies a fixed `+10 h` to
reach the AEST analysis frame. If that assumption is wrong, everything downstream still
runs: you get coefficients, a counterfactual, and a curtailment number. All of it
meaningless, and nothing raises.

This was hit for real while testing: with the frames mismatched, `GHI_cs > 0` and
`P_kw_norm_cs > 0.2` never co-occurred and the training set came out empty. That is the
*loud* failure. A one- or two-hour offset would not empty anything — it would quietly
bias every counterfactual. So read the **peak-hour gap**, not just the row counts.

In [10]:
ctf.build_structured(con, config)
se_store.register_store_views(con)

alignment = ctf.check_ghi_alignment(con)
display(alignment)

GHI vs generation diurnal alignment: ALIGNED
  correlation over hour-of-day : 0.999   (want > 0.8)
  GHI peaks at 12:00 AEST, generation at 12:00 AEST  -> gap 0 h   (want <= 1)


,hour_aest,mean_GHI,mean_P_norm,n
0,4,5.564206,0.001652,168809
1,5,43.634709,0.023992,1803045
2,6,112.046310,0.076894,4284860
3,7,216.027784,0.175931,6144503
4,8,352.098105,0.319646,6727151
5,9,493.138298,0.470876,6721616
6,10,596.706910,0.577492,6763131
7,11,652.556012,0.632691,6760134
8,12,658.448932,0.641889,5645372
9,13,618.951269,0.609716,6764560


### 5.1 Fit the model, gate it, apply it

`fit_ghi_model` regresses `P_norm / P_norm_cs = a + b·(GHI / GHI_cs)` with `a = 1 − b`,
forcing the line through (1, 1): on a clear-sky interval both ratios are 1 by
construction. One free parameter, which is what keeps it stable on small per-bin samples.

`mape_quality_gate` then drops sites the model cannot predict. Expect it to reject
**large-area postcodes preferentially** — that is correct behaviour, not random attrition,
and it means coverage is biased toward dense urban postcodes. `postcode_area_km2` travels
through to the gate report so the bias stays auditable.

In [11]:
model = ctf.fit_ghi_model(con)
se_store.register_store_views(con)
print(f"Model rows (site x time-of-day bin): {len(model):,}")

gate = ctf.mape_quality_gate(con)
print(f"Sites evaluated: {len(gate):,}   passing MAPE gate: {int(gate.passes_gate.sum()):,}")
display(gate.head())

# Is the gate rejecting big rural postcodes, as predicted?
# `.describe()` only emits 25/50/75% unless you ask, hence percentiles=.
if (len(gate) and "postcode_area_km2" in gate.columns
        and gate.postcode_area_km2.notna().any()):
    area_by_verdict = (
        gate.groupby("passes_gate").postcode_area_km2
            .describe(percentiles=[0.5, 0.9])
            .rename(columns={"50%": "median_km2", "90%": "p90_km2"})
            [["count", "median_km2", "p90_km2"]]
            .round(1)
    )
    area_by_verdict.index = area_by_verdict.index.map(
        {True: "passed gate", False: "rejected"})
    display(area_by_verdict)

    passed = gate.loc[gate.passes_gate, "postcode_area_km2"].median()
    failed = gate.loc[~gate.passes_gate, "postcode_area_km2"].median()
    if pd.notna(passed) and pd.notna(failed) and passed > 0:
        print(f"Median postcode area — rejected {failed:,.0f} km^2 vs "
              f"passed {passed:,.0f} km^2  ({failed / passed:.1f}x)")
        print("Expected: irradiance averaged over a large polygon is a poor proxy for")
        print("one roof, so the model fits worst where the postcode is biggest. This is")
        print("a coverage bias toward urban sites, not random attrition — state it.")
else:
    print("No geography on the gate report; run section 2 (attach_geography) first.")

Model rows (site x time-of-day bin): 127,297
Sites evaluated: 1,600   passing MAPE gate: 1,600


,site_alias,n_eval,mape,passes_gate,postcode_area_km2
0,AUS1024,22154,0.2594,True,18.8
1,AUS013,30680,0.2074,True,12.0
2,AUS015,26953,0.2548,True,3.3
3,AUS1094,29238,0.1452,True,13.8
4,AUS1046,27882,0.2042,True,974.8


,count,median_km2,p90_km2
passes_gate,,,
passed gate,1600.0,24.3,1029.4


In [12]:
built = ctf.build_uncurtailedpv(con, gate)
se_store.register_store_views(con)
display(built)

coverage = ctf.counterfactual_coverage(con, config)
display(coverage.head(20))

,n_rows,n_sites
0,44621397,1600


,postcode_size,n_sites,n_with_counterfactual,pct_covered
0,a. < 25 km2,805,804,99.88
1,b. 25-100,297,297,100.00
2,c. 100-1000,333,333,100.00
3,d. > 1000 km2,167,166,99.40


### What this unblocks

With `se_uncurtailedpv` written:

- **notebook 03 section 5b** — Volt-Watt *response-supported* conformance
- **notebook 06** — Method B, the GHI counterfactual attribution, and the evidence tiers

Read `pct_covered` before either. The counterfactual only covers sites that passed the
gate, and a low coverage number does not invalidate the result — it bounds what the
result is *about*.